# Notebook 2 — Exploratory Data Analysis

## What changed from R01

**EDA now runs on the training pool only.** The frozen 80/20 split happens
first, and the 200 test rows are not described, plotted or correlated here.

This is stricter than the examination required, and worth a sentence in M6.
Analyst-level peeking is a real leakage channel: if you choose composite
weights, or notice an outlier, or decide a feature matters after looking at a
plot that included the test rows, that decision carries test information into
the model even though no `.fit()` touched them. Since GATE-1 already failed on
mechanical leakage, the cheapest way to make the corrected pipeline credible
is for nothing at all to have looked at the test partition before Notebook 8.

Also new here: the plots that answer questions the examination asked and R01
had no figure for — school-level dropout rates, the income-category audit,
the attendance out-of-range distribution, and where the base rate sits.

In [ ]:
# ---- bootstrap: repo-relative imports, no drive.mount, no hard-coded path ----
import sys, os
from pathlib import Path

def _find_repo(start=None):
    p = Path(start or Path.cwd()).resolve()
    for c in [p, *p.parents]:
        if (c / "config.py").exists():
            return c
    return p

REPO = Path(os.environ["DROPOUT_REPO"]) if os.environ.get("DROPOUT_REPO") else _find_repo()
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

# In Colab, clone the repo first, then run:
#     import os; os.environ["DROPOUT_REPO"] = "/content/student-dropout-prediction-ghana"
# Raw pupil-level data is NOT in the repo (ethics); place it under data-raw/
# locally. Nothing below calls drive.mount().

import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
from config import *
from pipeline import frozen_split, binarise_target

banner("NOTEBOOK 2 — EDA (TRAINING POOL ONLY)")
OUT = run_dir("notebook02_eda")
FIGS = OUT / "figures"
print("outputs ->", OUT)

df = pd.read_csv(CLEANED_CSV)
train_pool, test_holdout = frozen_split(df)

print(f"full           : {len(df)} rows")
print(f"train_pool     : {len(train_pool)} rows, "
      f"{int(train_pool[TARGET].sum())} dropout "
      f"({100*train_pool[TARGET].mean():.1f}%)")
print(f"test_holdout   : {len(test_holdout)} rows "
      "-- NOT described below, not plotted, not touched until Notebook 8")

eda = train_pool          # everything from here uses eda, never df
BASE_RATE = float(eda[TARGET].mean())

In [ ]:
# ---- 1. shape, dtypes, target -------------------------------------------
print(eda.dtypes.value_counts().to_string())
num = [c for c in eda.columns if not is_text(eda[c])]
cat = [c for c in eda.columns if c not in num]
print(f"\nnumeric: {len(num)}   categorical: {len(cat)}")

desc = eda[num].describe().T
desc.to_csv(OUT / "descriptive_statistics.csv")
print("\nTable 2 source (report min/max honestly — out-of-range values are "
      "visible here and treated in-fold):")
print(desc[["mean", "std", "min", "max"]].round(3).to_string())

fig, ax = plt.subplots(figsize=(5, 4))
sns.countplot(data=eda, x=TARGET, ax=ax)
ax.set_title(f"Target distribution (training pool, base rate {100*BASE_RATE:.1f}%)")
for p in ax.patches:
    ax.annotate(f"{int(p.get_height())}", (p.get_x()+p.get_width()/2, p.get_height()),
                ha="center", va="bottom")
plt.tight_layout(); plt.savefig(FIGS / "target_distribution.png", dpi=200); plt.close()
print(f"\nabsolute positive count in the training pool: {int(eda[TARGET].sum())}")
print("Carry this number beside every reported rate (J4).")

In [ ]:
# ---- 2. school-level structure (GATE-1 iv, Q2) --------------------------
if SCHOOL_COL in eda.columns:
    g = (eda.groupby(SCHOOL_COL)
           .agg(n=(TARGET, "size"), n_dropout=(TARGET, "sum")))
    g["rate_pct"] = (100 * g["n_dropout"] / g["n"]).round(1)
    g = g.sort_values("n", ascending=False)
    g.to_csv(OUT / "school_structure.csv")
    print(g.to_string())

    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    g["n"].plot.bar(ax=axes[0], color="steelblue")
    axes[0].axhline(len(eda)/len(g), ls="--", c="k", lw=1, label="even split")
    axes[0].set_title("Records per school"); axes[0].legend()
    g["rate_pct"].plot.bar(ax=axes[1], color="indianred")
    axes[1].axhline(100*BASE_RATE, ls="--", c="k", lw=1, label="overall base rate")
    axes[1].set_title("Dropout rate per school (%)"); axes[1].legend()
    plt.tight_layout(); plt.savefig(FIGS / "school_structure.png", dpi=200); plt.close()

    print(f"\n{g.shape[0]} schools. Largest holds {100*g['n'].iloc[0]/len(eda):.0f}%. "
          f"Rates span {g['rate_pct'].min():.1f}%-{g['rate_pct'].max():.1f}%.")
    print("This is the figure to put in your own limitations paragraph before "
          "a reviewer finds it in the data file.")

In [ ]:
# ---- 3. income category audit (Q3) -------------------------------------
inc = SOCIOECONOMIC_COLS["family_income"]
if inc in eda.columns:
    vc = eda[inc].value_counts(dropna=False)
    print(f"{inc} as recorded:\n{vc.to_string()}")
    print(f"\nafter canonicalisation -> {sorted(set(CATEGORY_CANONICAL[inc].values()))}")
    print(f"ordinal map -> {ORDINAL_MAPS.get(inc)}")
    print("\n'Unknown' maps to NaN plus a separate indicator, NOT to Medium. "
          "The R01 Notebook 3 mapped \"don't know\" to 1 (Medium), which "
          "silently imputes a value and hides the non-response.")

    ct = pd.crosstab(eda[inc], eda[TARGET], normalize="index") * 100
    print("\ndropout % by recorded income category:")
    print(ct.round(1).to_string())

In [ ]:
# ---- 4. attendance distributions and the >100% cases -------------------
present = [c for c in ATTENDANCE_COLS if c in eda.columns]
if present:
    fig, axes = plt.subplots(1, len(present), figsize=(4*len(present), 3.5))
    axes = np.atleast_1d(axes)
    for ax, c in zip(axes, present):
        s = pd.to_numeric(eda[c], errors="coerce")
        sns.histplot(s, ax=ax, bins=30)
        ax.axvline(ATTENDANCE_MAX, ls="--", c="r", lw=1)
        n_over = int((s > ATTENDANCE_MAX).sum())
        ax.set_title(f"{c}\n{n_over} values > 100%")
    plt.tight_layout(); plt.savefig(FIGS / "attendance_distributions.png", dpi=200)
    plt.close()
    for c in present:
        s = pd.to_numeric(eda[c], errors="coerce")
        print(f"{c:22s} min={s.min():7.1f} max={s.max():7.1f} "
              f">100: {int((s>ATTENDANCE_MAX).sum()):4d}")

In [ ]:
# ---- 5. correlations and the near-perfect-separation question ----------
# The sharpest thing in the examination: a model at 0.995 accuracy with one
# error on 200 pupils does not look like any real dropout problem. Before
# modelling, check whether a single raw feature already separates the classes.
corr = eda[num].corr(numeric_only=True)
plt.figure(figsize=(12, 9))
sns.heatmap(corr, cmap="coolwarm", center=0, square=False)
plt.title("Correlation matrix (training pool)")
plt.tight_layout(); plt.savefig(FIGS / "correlation_matrix.png", dpi=200); plt.close()

tc = (corr[TARGET].drop(TARGET).abs().sort_values(ascending=False))
print("absolute correlation with the target, top 15:")
print(tc.head(15).round(3).to_string())
tc.to_csv(OUT / "target_correlations.csv")

print("\nSINGLE-FEATURE SEPARATION CHECK")
print("AUC-PR achievable from each feature alone (training pool):")
from sklearn.metrics import average_precision_score
single = []
y = eda[TARGET].to_numpy()
for c in num:
    if c == TARGET:
        continue
    s = pd.to_numeric(eda[c], errors="coerce").fillna(eda[c].median())
    ap = max(average_precision_score(y, s), average_precision_score(y, -s))
    single.append({"feature": c, "auc_pr_alone": ap})
single = pd.DataFrame(single).sort_values("auc_pr_alone", ascending=False)
single["base_rate"] = BASE_RATE
single.to_csv(OUT / "single_feature_separation.csv", index=False)
print(single.head(12).round(4).to_string(index=False))
print(f"\nbase rate (an uninformative feature scores about this): {BASE_RATE:.4f}")
print("\nIf one feature alone reaches ~0.9 AUC-PR, that feature is carrying the "
      "separation and the modelling contribution is small. THAT is the finding "
      "to chase, and it is a better paper than a null on a loss function. "
      "If it is school_code, you have a cluster artefact, not a pupil-risk model.")

In [ ]:
# ---- 6. save -----------------------------------------------------------
write_manifest(OUT, {
    "notebook": "02_eda",
    "scope": "training pool only; test partition untouched",
    "train_pool_n": int(len(train_pool)),
    "train_pool_positive": int(train_pool[TARGET].sum()),
    "base_rate": BASE_RATE,
    "top_single_feature": single.iloc[0].to_dict() if len(single) else None,
})
print("figures ->", FIGS)
print("NEXT: Notebook 3 (feature engineering as a fold-safe pipeline).")